In [1]:
!pip -q install rank-bm25 underthesea qdrant-client
!pip -q install "transformers==4.51.3" "sentence-transformers==3.0.1" "accelerate==1.6.0" "FlagEmbedding==1.2.11"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 75.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.1/147.1 kB 9.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 114.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have hu

# Configuration

In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from google.colab import userdata
from underthesea import word_tokenize
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from FlagEmbedding import FlagReranker
from qdrant_client import QdrantClient

MODE = "dev"  # "dev" or "test"

DEV_PATH = "/content/dev_dataset.jsonl"
TEST_PATH = "/content/test_dataset.jsonl"
CORPUS_PATH = "/content/full_DVSK_data_aiteamvn_256.json"

OUTPUT_ROOT = Path("/content/retrieval_outputs")
OUTPUT_DIR = OUTPUT_ROOT / MODE

QDRANT_COLLECTION = "dvsktt_aiteamvn_chunks_256"
DENSE_MODEL_NAME = "AITeamVN/Vietnamese_Embedding"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"

# top 20 from the first stage retrieval (BM25 and dense) will be used for re-ranking
FIRST_STAGE_TOP_K = 20
ALPHAS = [0.3, 0.5, 0.7]

RERANK_ALPHA = 0.5

RERANK_INPUT_TOP_K = 20
RERANK_OUTPUT_TOP_K = 10

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if MODE == "dev":
    DATASET_PATH = DEV_PATH
    EFFECTIVE_ALPHAS = ALPHAS
elif MODE == "test":
    DATASET_PATH = TEST_PATH
    EFFECTIVE_ALPHAS = [RERANK_ALPHA]
else:
    raise ValueError(
        f"MODE không hợp lệ: {MODE}. Chỉ chấp nhận 'dev' hoặc 'test'."
    )

def alpha_tag(alpha: float) -> str:
    return f"alpha{int(round(alpha * 10)):02d}"

# Load dataset and corpus

In [3]:
def read_jsonl(fname):
    with open(fname, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def read_json_list(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

dataset_rows = read_jsonl(DATASET_PATH)
corpus_rows = read_json_list(CORPUS_PATH)

# chuyển thành dict để tra cứu nhanh hơn
corpus_by_id = {str(row["chunk_id"]): row for row in corpus_rows}
chunk_ids = list(corpus_by_id.keys())

print(f"Dataset rows: {len(dataset_rows)}")
print(f"Corpus chunks: {len(corpus_rows)}")
print(f"Corpus chunks: {len(corpus_by_id)}")

Dataset rows: 264
Corpus chunks: 5343
Corpus chunks: 5343


# Connect to Qdrant

In [4]:
qdrant_url = userdata.get("QDRANT_URL")
qdrant_api_key = userdata.get("QDRANT_API_KEY")

qdrant_client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key,
    timeout=120,
)

qdrant_count = qdrant_client.count(
    collection_name=QDRANT_COLLECTION,
    exact=True,
).count

print(f"Qdrant collection: {QDRANT_COLLECTION}")
print(f"Qdrant points: {qdrant_count}")

Qdrant collection: dvsktt_aiteamvn_chunks_256
Qdrant points: 5343


# Build BM25 index from raw text

In [5]:
bm25_corpus_tokens = []

for cid in tqdm(chunk_ids, desc="Tokenizing corpus for BM25"):
    raw_text = corpus_by_id[cid]["raw_text"]
    tokens = word_tokenize(raw_text, format="text").split()
    bm25_corpus_tokens.append(tokens)

bm25 = BM25Okapi(bm25_corpus_tokens)

print("BM25 index is ready.")

Tokenizing corpus for BM25:   0%|          | 0/5343 [00:00<?, ?it/s]

BM25 index is ready.


# Load dense model and reranker

In [6]:
dense_model = SentenceTransformer(DENSE_MODEL_NAME,)

reranker = FlagReranker(
    RERANKER_MODEL_NAME,
    use_fp16=torch.cuda.is_available(),
)

print(f"Dense model: {DENSE_MODEL_NAME}")
print(f"Reranker model: {RERANKER_MODEL_NAME}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/708 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Dense model: AITeamVN/Vietnamese_Embedding
Reranker model: BAAI/bge-reranker-v2-m3


# Retrieval helpers

In [10]:
def make_evidence_item(cid, rank, extra_scores=None):

    item = {
        "rank": rank,
        "chunk_id": str(cid),
    }

    for key, value in extra_scores.items():
        item[key] = float(value)

    return item

def build_output_row(row, method_name, evidence_items):
    output = {
        "claim_id": row["claim_id"],
        "evidence_ids": row["evidence_ids"],
        "retrieval_method": method_name,
        "retrieved_evidence": evidence_items
    }
    return output

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def minmax_normalize(score_map):
    values = np.array(list(score_map.values()), dtype=np.float32)
    min_value = float(values.min())
    max_value = float(values.max())
    if max_value - min_value < 1e-12:
        return {key: 0.0 for key in score_map}
    return {
        key: float((value - min_value) / (max_value - min_value))
        for key, value in score_map.items()
    }

def query_qdrant(vector, top_k):
    result = qdrant_client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=vector.tolist(),
        limit=top_k,
        with_payload=False,
        with_vectors=False,
    )

    points = result.points if hasattr(result, "points") else result
    hits = []

    for point in points:
        hits.append((str(point.id), float(point.score)))

    return hits

# Run BM25 and dense retrieval

In [11]:
claims = [row["claim"] for row in dataset_rows]

bm25_hits_by_claim_id = {}
dense_hits_by_claim_id = {}

query_vectors = dense_model.encode(
    claims,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print(f"dataset rows {dataset_rows[:5]}")
print(f"\n query vectors {query_vectors[:5]}")

for row, query_vector in tqdm(zip(dataset_rows, query_vectors), total=len(dataset_rows), desc="Running BM25 and dense retrieval"):
    claim_id = str(row["claim_id"])
    claim = row["claim"]

    query_tokens = word_tokenize(claim, format="text").split()
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_top_indices = np.argsort(bm25_scores)[::-1][:FIRST_STAGE_TOP_K]
    bm25_hits = [(chunk_ids[idx], float(bm25_scores[idx])) for idx in bm25_top_indices]

    dense_hits = query_qdrant(query_vector, FIRST_STAGE_TOP_K)

    bm25_hits_by_claim_id[claim_id] = bm25_hits
    dense_hits_by_claim_id[claim_id] = dense_hits


print(f"kết quả bm25 {list(bm25_hits_by_claim_id.items())[:5]}")
print(f"kết quả dense {list(dense_hits_by_claim_id.items())[:5]}")

print("BM25 and dense retrieval are finished.")

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

dataset rows [{'claim_id': 'claim_000013', 'chunk_id': '019f17e1-0a34-793b-b170-d28a55c14360', 'pages': [7], 'raw_text': 'Cao Vương nhà Đường [tức Cao Biền] dẹp nước Nam Chiếu, khi đưa quân về qua châu Vũ Ninh, đêm nằm chiêm bao thấy có người lạ tự xưng là Cao Lỗ, nói: "Ngày xưa giúp An Dương Vương, có công đánh lui giặc, bị Lạc hầu gièm pha, phải bỏ đi, sau khi chết, trời thương không có tội gì, ban cho một dải núi sông này, cho làm chức quản lĩnh đô thống tướng quân, làm chủ mọi việc đánh dẹp giặc giã và mùa màng cày cấy.', 'claim': 'Cao Biền từng mơ thấy một nhân vật tự nhận là Cao Lỗ khi ông đang trên đường dẫn quân về qua địa phận châu Vũ Ninh.', 'label': 'SUPPORTED', 'claim_type': 'PERSON_ROLE', 'evidence_ids': ['019f17e1-0a34-793b-b170-d28a55c14360'], 'explanation': 'Nguồn ghi Cao Vương (Cao Biền) dẹp Nam Chiếu, khi về qua châu Vũ Ninh nằm chiêm bao thấy người lạ tự xưng là Cao Lỗ. Claim diễn đạt lại sự kiện này một cách khách quan.', 'split': 'dev'}, {'claim_id': 'claim_000014'

Running BM25 and dense retrieval:   0%|          | 0/264 [00:00<?, ?it/s]

kết quả bm25 [('claim_000013', [('019f17e1-0a34-793b-b170-d28a55c14360', 37.72144216920839), ('019f17e7-1bd0-7df5-bc1a-7ca04594ed0d', 30.44915147412111), ('019f17e1-adc8-7e11-bb3f-df23cbb0a0b4', 22.14485612530212), ('019f17e4-5374-7be4-b94b-d104eb8aa22e', 19.831477983600276), ('019f17e4-742b-7795-aa79-0c210a39099d', 19.728114308182445), ('019f17e4-697b-7f40-892c-6dabc8e08fcf', 19.231750861314236), ('019f17e2-c22e-78e4-9af7-fa9bcd4f7a13', 19.217232289318382), ('019f17e4-a016-7cad-92c1-38a88883f1d9', 18.495268581691956), ('019f17e5-c116-7694-8ec6-fcf05cee9cdd', 18.10730672246025), ('019f17eb-2e75-7f43-8d51-cfe5c80900e7', 17.520068867675814), ('019f17e9-89f1-7375-bb39-241e0401501e', 17.378940515053365), ('019f17e2-5884-7c6c-a536-e20eb5a73d3c', 17.136068726377147), ('019f17e3-bb53-75c7-be5c-820ec5a8cc5e', 16.85869377393145), ('019f17e4-7401-7af2-8e5d-8ce9728f757d', 16.835589473663763), ('019f17e4-7413-74cd-a327-1fbd2c77ece0', 16.802835553915692), ('019f17e6-515e-7605-88a8-59d8929a433c', 16

# Save BM25 and dense outputs

In [12]:
method_outputs = {}

for method_name, hits_by_claim_id in [
    ("bm25", bm25_hits_by_claim_id),
    ("dense", dense_hits_by_claim_id),
]:
    top20_rows = []

    for row in dataset_rows:
        claim_id = str(row["claim_id"])
        hits = hits_by_claim_id[claim_id]

        top20_evidence = [
            make_evidence_item(cid, rank=index + 1, extra_scores={f"{method_name}_score": score})
            for index, (cid, score) in enumerate(hits[:20])
        ]

        top20_rows.append(build_output_row(row, f"{method_name}_top20", top20_evidence))

    method_outputs[f"{method_name}_top20"] = top20_rows

    write_jsonl(OUTPUT_DIR / f"{method_name}_top20_all_labels_{MODE}.jsonl", top20_rows)

print("BM25 and dense output files are saved.")

BM25 and dense output files are saved.


# Save hybrid outputs

In [13]:
hybrid_hits_by_alpha = {}

for alpha in EFFECTIVE_ALPHAS:
    tag = alpha_tag(alpha)
    hybrid_hits_by_claim_id = {}
    top20_rows = []

    for row in tqdm(dataset_rows, desc=f"Building hybrid retrieval {tag}"):
        claim_id = str(row["claim_id"])
        bm25_hits = dict(bm25_hits_by_claim_id[claim_id])
        dense_hits = dict(dense_hits_by_claim_id[claim_id])

        merged_ids = sorted(set(bm25_hits) | set(dense_hits))

        bm25_raw = {cid: float(bm25_hits.get(cid, 0.0)) for cid in merged_ids}
        dense_raw = {cid: float(dense_hits.get(cid, 0.0)) for cid in merged_ids}

        bm25_norm = minmax_normalize(bm25_raw)
        dense_norm = minmax_normalize(dense_raw)

        scored = []
        for cid in merged_ids:
            hybrid_score = alpha * dense_norm[cid] + (1.0 - alpha) * bm25_norm[cid]
            scored.append({
                "chunk_id": cid,
                "hybrid_score": float(hybrid_score),
                "bm25_score": bm25_raw[cid],
                "dense_score": dense_raw[cid],
                "bm25_norm": bm25_norm[cid],
                "dense_norm": dense_norm[cid],
            })

        scored = sorted(scored, key=lambda item: item["hybrid_score"], reverse=True)[:FIRST_STAGE_TOP_K]
        hybrid_hits_by_claim_id[claim_id] = scored

        top20_evidence = [
            make_evidence_item(
                item["chunk_id"],
                rank=index + 1,
                extra_scores={
                    "hybrid_score": item["hybrid_score"],
                    "bm25_score": item["bm25_score"],
                    "dense_score": item["dense_score"],
                    "bm25_norm": item["bm25_norm"],
                    "dense_norm": item["dense_norm"],
                },
            )
            for index, item in enumerate(scored)
        ]

        top20_rows.append(build_output_row(row, f"hybrid_{tag}_top20", top20_evidence))

    hybrid_hits_by_alpha[tag] = hybrid_hits_by_claim_id
    method_outputs[f"hybrid_{tag}_top20"] = top20_rows

    write_jsonl(OUTPUT_DIR / f"hybrid_{tag}_top20_all_labels_{MODE}.jsonl", top20_rows)

print("Hybrid output files are saved.")

Building hybrid retrieval alpha03:   0%|          | 0/264 [00:00<?, ?it/s]

Building hybrid retrieval alpha05:   0%|          | 0/264 [00:00<?, ?it/s]

Building hybrid retrieval alpha07:   0%|          | 0/264 [00:00<?, ?it/s]

Hybrid output files are saved.


# Save reranker outputs

In [14]:
rerank_tag = alpha_tag(RERANK_ALPHA)

if rerank_tag not in hybrid_hits_by_alpha:
    raise ValueError(
        f"RERANK_ALPHA={RERANK_ALPHA} was not generated in this run. "
        f"Generated alpha tags: {list(hybrid_hits_by_alpha.keys())}"
    )

reranker_top10_rows = []

for row in tqdm(dataset_rows, desc=f"Reranking hybrid candidates {rerank_tag}"):
    claim_id = str(row["claim_id"])
    claim = row["claim"]
    candidates = hybrid_hits_by_alpha[rerank_tag][claim_id][:RERANK_INPUT_TOP_K]

    pairs = []
    for item in candidates:
        raw_text = corpus_by_id[item["chunk_id"]]["raw_text"]
        pairs.append([claim, raw_text])

    rerank_scores = reranker.compute_score(pairs)

    if isinstance(rerank_scores, float):
        rerank_scores = [rerank_scores]

    reranked = []
    for item, rerank_score in zip(candidates, rerank_scores):
        reranked.append({
            **item,
            "reranker_score": float(rerank_score),
        })

    reranked = sorted(reranked, key=lambda item: item["reranker_score"], reverse=True,)[:RERANK_OUTPUT_TOP_K]

    top10_evidence = [
        make_evidence_item(
            item["chunk_id"],
            rank=index + 1,
            extra_scores={
                "reranker_score": item["reranker_score"],
                "hybrid_score": item["hybrid_score"],
                "bm25_score": item["bm25_score"],
                "dense_score": item["dense_score"],
                "bm25_norm": item["bm25_norm"],
                "dense_norm": item["dense_norm"],
            },
        )
        for index, item in enumerate(reranked)
    ]

    reranker_top10_rows.append(build_output_row(row, f"hybrid_reranker_{rerank_tag}_top10", top10_evidence))

method_outputs[f"hybrid_reranker_{rerank_tag}_top10"] = reranker_top10_rows

write_jsonl(OUTPUT_DIR / f"hybrid_reranker_{rerank_tag}_top10_all_labels_{MODE}.jsonl", reranker_top10_rows)

print("Reranker output files are saved.")

Reranking hybrid candidates alpha05:   0%|          | 0/264 [00:00<?, ?it/s]

Reranker output files are saved.


# Save sanity summary and configuration

In [16]:
import shutil
from google.colab import files

archive_name = f"retrieval_outputs_{MODE}"

shutil.make_archive(archive_name, "zip", root_dir=OUTPUT_DIR)

files.download(f"{archive_name}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>